Step -1 Define Schema as STRING

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
%sql
create catalog retail_project

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_project.raw_data;

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS retail_project.raw_data.raw_file
LOCATION 'abfss://retail@retaildomain.dfs.core.windows.net/raw_data/raw_file';

Step -2 Read Raw as CSV

In [0]:
orders_bronze_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(orders_schema)
    .load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")
)

customers_bronze_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")
)

In [0]:
customers_bronze_df.display()

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,Active
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,Active
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,Inactive
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,Inactive
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,Inactive
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,Active
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,Inactive
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,Active
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,Inactive
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,Active


In [0]:
orders_bronze_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status
O00001,13-02-2025,C0019,P210,Air Conditioner,Electronic,1,9314,0.05,null,Cash,Pune,Completed
O00002,22-01-2025,C0349,P207,Microwave Oven,Fashon,-1,abc,0.05,null,Credit Card,Chennai,Returned
O00003,2025-01-06,C0571,null,Refrigerator,electronics,-1,abc,0.05,null,Credit Card,Pune,Cancelled
O00004,2025-01-13,C0426,P201,iPhone 15,Fashon,3,61749,0.05,null,null,Pune,Completed
O00005,2025/02/24,C0640,P201,iPhone 15,electronics,3,abc,10%,null,Debit Card,Hyderabad,Returned
O00006,2025-02-16,C0157,P205,Levi Jeans,Fashion,3,55005,10%,null,Credit Card,Delhi,Cancelled
O00007,2025-01-15,C0336,P203,Sony Headphones,electronics,1,19012,0.1,null,Credit Card,Bangalore,completed
O00008,2025-03-12,C0597,P202,Samsung TV,Fashon,2,abc,0.1,null,null,Hyderabad,Cancelled
O00009,2025/03/26,C0103,P207,Microwave Oven,Electronics,1,abc,0,null,Credit Card,,Completed
O00010,2025-03-11,C0110,P208,Refrigerator,Fashon,1,abc,5%,null,Debit Card,Mumbai,Cancelled


Step - 3 Add Ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp


orders_bronze_df = orders_bronze_df.withColumn('ingest_timestamp', current_timestamp())
customers_bronze_df = customers_bronze_df.withColumn('ingest_timestamp',current_timestamp())

Step -4 Save AS Delta Table in Bronze

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.bronze")
orders_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_orders')
customers_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_customers')


Silver Layer Transformation
we will clean the retail orders and customer dataset by:

Fixing date formats

Standardizing category values

Removing duplicates

Handling null values

Converting discount from "5%" to 0.05 and "10%" to 0.1

Removing negative quantities

Fixing casing issues

Casting unit_price to decimal

Validating emails

Recalculating total_amount

-- and more

STEP 1: Load Bronze Orders

In [0]:

bronze_order_df = spark.read.table("retail_project.bronze.bronze_orders")

display(bronze_order_df.limit(100))

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,13-02-2025,C0019,P210,Air Conditioner,Electronic,1,9314,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00002,22-01-2025,C0349,P207,Microwave Oven,Fashon,-1,abc,0.05,null,Credit Card,Chennai,Returned,2026-11-10T13:30:01.911Z
O00003,2025-01-06,C0571,null,Refrigerator,electronics,-1,abc,0.05,null,Credit Card,Pune,Cancelled,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,Fashon,3,61749,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025/02/24,C0640,P201,iPhone 15,electronics,3,abc,10%,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,Fashion,3,55005,10%,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,electronics,1,19012,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,Fashon,2,abc,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025/03/26,C0103,P207,Microwave Oven,Electronics,1,abc,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,Fashon,1,abc,5%,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z


STEP 2: Fix Date Formats

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
    coalesce(
        try_to_date(col('order_date'),'yyyy-MM-dd'),
        try_to_date(col('order_date'),'dd-MM-yyyy'),
        try_to_date(col('order_date'),'yyyy/MM/dd'),
        try_to_date(col('order_date'),'dd/MM/yyyy')
    )
)
display(silver_df.limit(100))

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,Electronic,1,9314,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00002,2025-01-22,C0349,P207,Microwave Oven,Fashon,-1,abc,0.05,null,Credit Card,Chennai,Returned,2026-11-10T13:30:01.911Z
O00003,2025-01-06,C0571,null,Refrigerator,electronics,-1,abc,0.05,null,Credit Card,Pune,Cancelled,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,Fashon,3,61749,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,electronics,3,abc,10%,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,Fashion,3,55005,10%,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,electronics,1,19012,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,Fashon,2,abc,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,Electronics,1,abc,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,Fashon,1,abc,5%,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z


STEP 3: Standardize Category Values

In [0]:
silver_df.select('category').distinct().display()

category
Electronic
Fashon
electronics
Fashion
Electronics
Home & Living
Home and Living


In [0]:
from pyspark.sql.functions import upper, trim,when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

In [0]:
from pyspark.sql.functions import count ,countDistinct

silver_df.select(count('order_id'),countDistinct('order_id')).display()

count(order_id),count(DISTINCT order_id)
10050,10000


In [0]:

silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'),countDistinct('order_id')).display()

count(order_id),count(DISTINCT order_id)
10000,10000


STEP 5: Handle Null Values discount

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").isNull(), "0")
    .when(col("discount") == "ten", "0.1")
    .otherwise(col("discount")),
)

In [0]:

silver_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,ELECTRONICS,1,9314,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00002,2025-01-22,C0349,P207,Microwave Oven,FASHION,-1,abc,0.05,null,Credit Card,Chennai,Returned,2026-11-10T13:30:01.911Z
O00003,2025-01-06,C0571,null,Refrigerator,ELECTRONICS,-1,abc,0.05,null,Credit Card,Pune,Cancelled,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,FASHION,3,61749,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,ELECTRONICS,3,abc,10%,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,FASHION,3,55005,10%,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,ELECTRONICS,1,19012,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,FASHION,2,abc,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,ELECTRONICS,1,abc,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,FASHION,1,abc,5%,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z


STEP 6: Convert Discount from "5%" to Decimal
some rows contain:

5%

we need 0.05

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,ELECTRONICS,1,9314,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00002,2025-01-22,C0349,P207,Microwave Oven,FASHION,-1,abc,0.05,null,Credit Card,Chennai,Returned,2026-11-10T13:30:01.911Z
O00003,2025-01-06,C0571,null,Refrigerator,ELECTRONICS,-1,abc,0.05,null,Credit Card,Pune,Cancelled,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,FASHION,3,61749,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,ELECTRONICS,3,abc,0.1,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,FASHION,3,55005,0.1,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,ELECTRONICS,1,19012,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,FASHION,2,abc,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,ELECTRONICS,1,abc,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,FASHION,1,abc,0.05,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z


STEP 7: Remove Negative Quantity

In [0]:
silver_df = silver_df.filter(col("quantity")>=0)

In [0]:

silver_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,ELECTRONICS,1,9314,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,FASHION,3,61749,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,ELECTRONICS,3,abc,0.1,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,FASHION,3,55005,0.1,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,ELECTRONICS,1,19012,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,FASHION,2,abc,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,ELECTRONICS,1,abc,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,FASHION,1,abc,0.05,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z
O00011,2025-01-17,C0173,P203,Sony Headphones,HOME & LIVING,1,abc,0.05,null,null,Chennai,Returned,2026-11-10T13:30:01.911Z
O00012,2025-03-12,C0006,P209,Office Chair,HOME & LIVING,3,76588,0.05,null,Cash,Chennai,Cancelled,2026-11-10T13:30:01.911Z


STEP 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,ELECTRONICS,1,9314.00,0.05,null,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,FASHION,3,61749.00,0.05,null,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,ELECTRONICS,3,null,0.1,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,FASHION,3,55005.00,0.1,null,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,ELECTRONICS,1,19012.00,0.1,null,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,FASHION,2,null,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,ELECTRONICS,1,null,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,FASHION,1,null,0.05,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z
O00011,2025-01-17,C0173,P203,Sony Headphones,HOME & LIVING,1,null,0.05,null,null,Chennai,Returned,2026-11-10T13:30:01.911Z
O00012,2025-03-12,C0006,P209,Office Chair,HOME & LIVING,3,76588.00,0.05,null,Cash,Chennai,Cancelled,2026-11-10T13:30:01.911Z


STEP 9: Recalculate Total Amount

In [0]:
silver_df = silver_df.withColumn(
    "total_amount",
    col("quantity").cast("int")
    * col("unit_price")
    * (1 - col("discount").cast("double")),
)
  

In [0]:

silver_df.display()

order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00001,2025-02-13,C0019,P210,Air Conditioner,ELECTRONICS,1,9314.00,0.05,8848.3,Cash,Pune,Completed,2026-11-10T13:30:01.911Z
O00004,2025-01-13,C0426,P201,iPhone 15,FASHION,3,61749.00,0.05,175984.65,null,Pune,Completed,2026-11-10T13:30:01.911Z
O00005,2025-02-24,C0640,P201,iPhone 15,ELECTRONICS,3,null,0.1,null,Debit Card,Hyderabad,Returned,2026-11-10T13:30:01.911Z
O00006,2025-02-16,C0157,P205,Levi Jeans,FASHION,3,55005.00,0.1,148513.5,Credit Card,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00007,2025-01-15,C0336,P203,Sony Headphones,ELECTRONICS,1,19012.00,0.1,17110.8,Credit Card,Bangalore,completed,2026-11-10T13:30:01.911Z
O00008,2025-03-12,C0597,P202,Samsung TV,FASHION,2,null,0.1,null,null,Hyderabad,Cancelled,2026-11-10T13:30:01.911Z
O00009,2025-03-26,C0103,P207,Microwave Oven,ELECTRONICS,1,null,0,null,Credit Card,,Completed,2026-11-10T13:30:01.911Z
O00010,2025-03-11,C0110,P208,Refrigerator,FASHION,1,null,0.05,null,Debit Card,Mumbai,Cancelled,2026-11-10T13:30:01.911Z
O00011,2025-01-17,C0173,P203,Sony Headphones,HOME & LIVING,1,null,0.05,null,null,Chennai,Returned,2026-11-10T13:30:01.911Z
O00012,2025-03-12,C0006,P209,Office Chair,HOME & LIVING,3,76588.00,0.05,218275.8,Cash,Chennai,Cancelled,2026-11-10T13:30:01.911Z


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.silver")
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_orders")

Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project
STEP 1: Read Bronze Table

In [0]:
customers_bronze_df = spark.table("retail_project.bronze.bronze_customers")

display(customers_bronze_df)

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,Active,2026-11-10T13:30:07.283Z
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,Active,2026-11-10T13:30:07.283Z
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,Inactive,2026-11-10T13:30:07.283Z
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,Inactive,2026-11-10T13:30:07.283Z
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,Inactive,2026-11-10T13:30:07.283Z
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,Active,2026-11-10T13:30:07.283Z
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,Inactive,2026-11-10T13:30:07.283Z
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,Active,2026-11-10T13:30:07.283Z
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,Inactive,2026-11-10T13:30:07.283Z
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,Active,2026-11-10T13:30:07.283Z


STEP 2: Trim Spaces

In [0]:
from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_timestamp")
)

display(customers_df)
     

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,Active,2026-11-10T13:30:07.283Z
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,Active,2026-11-10T13:30:07.283Z
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,Inactive,2026-11-10T13:30:07.283Z
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,Inactive,2026-11-10T13:30:07.283Z
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,Inactive,2026-11-10T13:30:07.283Z
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,Active,2026-11-10T13:30:07.283Z
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,Inactive,2026-11-10T13:30:07.283Z
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,Active,2026-11-10T13:30:07.283Z
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,Inactive,2026-11-10T13:30:07.283Z
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,Active,2026-11-10T13:30:07.283Z


STEP 3: Remove Duplicate Customers

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(desc("ingest_timestamp"))

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = customers_df.filter("row_num = 1") \
                           .drop("row_num")

customers_df.display()

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,Active,2026-11-10T13:30:07.283Z
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,Active,2026-11-10T13:30:07.283Z
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,Inactive,2026-11-10T13:30:07.283Z
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,Inactive,2026-11-10T13:30:07.283Z
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,Inactive,2026-11-10T13:30:07.283Z
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,Active,2026-11-10T13:30:07.283Z
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,Inactive,2026-11-10T13:30:07.283Z
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,Active,2026-11-10T13:30:07.283Z
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,Inactive,2026-11-10T13:30:07.283Z
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,Active,2026-11-10T13:30:07.283Z


STEP 4: Fix and Standardize Status

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,ACTIVE,2026-11-10T13:30:07.283Z
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,ten,ACTIVE,2026-11-10T13:30:07.283Z
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,ten,INACTIVE,2026-11-10T13:30:07.283Z
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,INACTIVE,2026-11-10T13:30:07.283Z
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,INACTIVE,2026-11-10T13:30:07.283Z
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,ACTIVE,2026-11-10T13:30:07.283Z
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,INACTIVE,2026-11-10T13:30:07.283Z
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,ten,ACTIVE,2026-11-10T13:30:07.283Z
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,INACTIVE,2026-11-10T13:30:07.283Z
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,ten,ACTIVE,2026-11-10T13:30:07.283Z


STEP 5: Convert loyalty_points to Integer

In [0]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "").try_cast("int")
)


In [0]:

customers_df.display()

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,ACTIVE,2026-11-10T13:30:07.283Z
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,null,ACTIVE,2026-11-10T13:30:07.283Z
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,null,INACTIVE,2026-11-10T13:30:07.283Z
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,INACTIVE,2026-11-10T13:30:07.283Z
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,INACTIVE,2026-11-10T13:30:07.283Z
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,ACTIVE,2026-11-10T13:30:07.283Z
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,INACTIVE,2026-11-10T13:30:07.283Z
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,null,ACTIVE,2026-11-10T13:30:07.283Z
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,INACTIVE,2026-11-10T13:30:07.283Z
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,null,ACTIVE,2026-11-10T13:30:07.283Z


STEP 6: Validate Email Format

In [0]:

customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

customers_df.select("email", "email_valid").display()

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True).drop("email_valid")

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
/home/spark-cb4b1ea6-1ccf-4428-9093-21/.ipykernel/62/command-5441113091906331-1241857815:4: SyntaxWarning: invalid escape sequence '\.'
  col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'


email,email_valid
nehasharma@email.com,true
karangupta@email.com,true
snehaverma@email.com,true
anjalikapoor@email.com,true
snehasingh@email.com,true
karanmehta@email.com,true
rohansingh@email.com,true
amitrao@email.com,true
rohanrao@email.com,true
snehaiyer@email.com,true


STEP 7: Fix Date Formats

In [0]:
from pyspark.sql.functions import to_date, coalesce

customers_df = (customers_df.withColumn(
    "dob_parsed",
    coalesce(
        try_to_date(col("date_of_birth"), "yyyy-MM-dd"),
        try_to_date(col("date_of_birth"), "dd-MM-yyyy"),
        try_to_date(col("date_of_birth"), "yyyy/MM/dd"),
        try_to_date(col("date_of_birth"), "dd/MM/yyyy"),
    )
).withColumn(
    "registration_parsed",
    coalesce(
        try_to_date(col("registration_date"), "yyyy-MM-dd"),
        try_to_date(col("registration_date"), "dd-MM-yyyy"),
        try_to_date(col("registration_date"), "yyyy/MM/dd"),
        try_to_date(col("registration_date"), "dd/MM/yyyy"),
    )
)
)
customers_df.display()
     

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_points,status,ingest_timestamp,dob_parsed,registration_parsed
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,04-03-1991,Delhi,Delhi,06-04-2023,1608,ACTIVE,2026-11-10T13:30:07.283Z,1991-03-04,2023-04-06
C0002,Karan Gupta,karangupta@email.com,not_available,Male,06/08/1985,Jaipur,Rajasthan,12-10-2023,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-08-06,2023-10-12
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,1987/01/30,Jaipur,Maharashtra,2024/07/19,null,INACTIVE,2026-11-10T13:30:07.283Z,1987-01-30,2024-07-19
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,1997-12-13,Mumbai,Uttar Pradesh,12/10/2023,null,INACTIVE,2026-11-10T13:30:07.283Z,1997-12-13,2023-10-12
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,1993/04/21,Bangalore,West Bengal,2024/01/24,null,INACTIVE,2026-11-10T13:30:07.283Z,1993-04-21,2024-01-24
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,2000-05-10,Chennai,West Bengal,10/02/2024,569,ACTIVE,2026-11-10T13:30:07.283Z,2000-05-10,2024-02-10
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,1990/07/14,Mumbai,Delhi,23-04-2023,null,INACTIVE,2026-11-10T13:30:07.283Z,1990-07-14,2023-04-23
C0008,Amit Rao,amitrao@email.com,28499601,Male,18/05/1998,Kolkata,Telangana,2024/10/18,null,ACTIVE,2026-11-10T13:30:07.283Z,1998-05-18,2024-10-18
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,1988/07/19,Bangalore,Rajasthan,2023-06-15,1125,INACTIVE,2026-11-10T13:30:07.283Z,1988-07-19,2023-06-15
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,1985/06/09,Lucknow,Maharashtra,29-06-2024,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-06-09,2024-06-29


In [0]:
customers_df = customers_df.drop("date_of_birth", "registration_date") \
                           .withColumnRenamed("dob_parsed", "date_of_birth") \
                           .withColumnRenamed("registration_parsed", "registration_date")
     

In [0]:
customers_df.display()
     

customer_id,customer_name,email,phone,gender,city,state,loyalty_points,status,ingest_timestamp,date_of_birth,registration_date
C0001,Neha Sharma,nehasharma@email.com,not_available,Female,Delhi,Delhi,1608,ACTIVE,2026-11-10T13:30:07.283Z,1991-03-04,2023-04-06
C0002,Karan Gupta,karangupta@email.com,not_available,Male,Jaipur,Rajasthan,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-08-06,2023-10-12
C0003,Sneha Verma,snehaverma@email.com,14097807,Male,Jaipur,Maharashtra,null,INACTIVE,2026-11-10T13:30:07.283Z,1987-01-30,2024-07-19
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,Male,Mumbai,Uttar Pradesh,null,INACTIVE,2026-11-10T13:30:07.283Z,1997-12-13,2023-10-12
C0005,Sneha Singh,snehasingh@email.com,24678832,Male,Bangalore,West Bengal,null,INACTIVE,2026-11-10T13:30:07.283Z,1993-04-21,2024-01-24
C0006,Karan Mehta,karanmehta@email.com,not_available,Female,Chennai,West Bengal,569,ACTIVE,2026-11-10T13:30:07.283Z,2000-05-10,2024-02-10
C0007,Rohan Singh,rohansingh@email.com,24942594,Female,Mumbai,Delhi,null,INACTIVE,2026-11-10T13:30:07.283Z,1990-07-14,2023-04-23
C0008,Amit Rao,amitrao@email.com,28499601,Male,Kolkata,Telangana,null,ACTIVE,2026-11-10T13:30:07.283Z,1998-05-18,2024-10-18
C0009,Rohan Rao,rohanrao@email.com,12933722,Male,Bangalore,Rajasthan,1125,INACTIVE,2026-11-10T13:30:07.283Z,1988-07-19,2023-06-15
C0010,Sneha Iyer,snehaiyer@email.com,34516264,Male,Lucknow,Maharashtra,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-06-09,2024-06-29


STEP 8: Standardize Gender

In [0]:
customers_df = customers_df.withColumn(
    "gender",
    when(upper(col("gender")) == "MALE", "M")
    .when(upper(col("gender")) == "FEMALE", "F")
    .otherwise("U")
)

In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customers")
     

Gold Layer in Databricks – Building Fact & Dimension Tables
We will:

1️⃣ Create Dimension Tables

2️⃣ Create Fact Table

3️⃣ Join Orders & Customers

4️⃣ Build KPI-ready aggregates

5️⃣ Store optimized Gold tables

STEP 1: Load Silver Tables

In [0]:
customers_df = spark.read.table("retail_project.silver.silver_customers")
orders_df = spark.read.table("retail_project.silver.silver_orders")

customers_df.display()
orders_df.display()

customer_id,customer_name,email,phone,gender,city,state,loyalty_points,status,ingest_timestamp,date_of_birth,registration_date
C0001,Neha Sharma,nehasharma@email.com,not_available,F,Delhi,Delhi,1608,ACTIVE,2026-11-10T13:30:07.283Z,1991-03-04,2023-04-06
C0002,Karan Gupta,karangupta@email.com,not_available,M,Jaipur,Rajasthan,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-08-06,2023-10-12
C0003,Sneha Verma,snehaverma@email.com,14097807,M,Jaipur,Maharashtra,null,INACTIVE,2026-11-10T13:30:07.283Z,1987-01-30,2024-07-19
C0004,Anjali Kapoor,anjalikapoor@email.com,not_available,M,Mumbai,Uttar Pradesh,null,INACTIVE,2026-11-10T13:30:07.283Z,1997-12-13,2023-10-12
C0005,Sneha Singh,snehasingh@email.com,24678832,M,Bangalore,West Bengal,null,INACTIVE,2026-11-10T13:30:07.283Z,1993-04-21,2024-01-24
C0006,Karan Mehta,karanmehta@email.com,not_available,F,Chennai,West Bengal,569,ACTIVE,2026-11-10T13:30:07.283Z,2000-05-10,2024-02-10
C0007,Rohan Singh,rohansingh@email.com,24942594,F,Mumbai,Delhi,null,INACTIVE,2026-11-10T13:30:07.283Z,1990-07-14,2023-04-23
C0008,Amit Rao,amitrao@email.com,28499601,M,Kolkata,Telangana,null,ACTIVE,2026-11-10T13:30:07.283Z,1998-05-18,2024-10-18
C0009,Rohan Rao,rohanrao@email.com,12933722,M,Bangalore,Rajasthan,1125,INACTIVE,2026-11-10T13:30:07.283Z,1988-07-19,2023-06-15
C0010,Sneha Iyer,snehaiyer@email.com,34516264,M,Lucknow,Maharashtra,null,ACTIVE,2026-11-10T13:30:07.283Z,1985-06-09,2024-06-29


order_id,order_date,customer_id,product_id,product_name,category,quantity,unit_price,discount,total_amount,payment_method,store_location,order_status,ingest_timestamp
O00079,2025-01-29,null,P201,iPhone 15,FASHION,1,null,0.1,null,null,Jaipur,Returned,2026-11-10T13:30:01.911Z
O00099,2025-01-11,C0664,P203,Sony Headphones,FASHION,3,45265.00,0,135795.0,Cash,Delhi,Returned,2026-11-10T13:30:01.911Z
O00124,2025-01-12,C0354,P201,iPhone 15,ELECTRONICS,2,null,0.1,null,Cash,Lucknow,Cancelled,2026-11-10T13:30:01.911Z
O00171,2025-01-14,C0500,P205,Levi Jeans,HOME & LIVING,2,null,0.1,null,Cash,Bangalore,Cancelled,2026-11-10T13:30:01.911Z
O00189,2025-02-12,C0345,P205,Levi Jeans,HOME & LIVING,1,null,0,null,Cash,Bangalore,completed,2026-11-10T13:30:01.911Z
O00216,2025-01-26,C0792,P209,Office Chair,HOME & LIVING,2,null,0.1,null,null,,Returned,2026-11-10T13:30:01.911Z
O00245,2025-02-05,C0792,P201,iPhone 15,ELECTRONICS,2,76887.00,0,153774.0,Cash,Delhi,Cancelled,2026-11-10T13:30:01.911Z
O00254,2025-02-12,C0814,P203,Sony Headphones,ELECTRONICS,1,null,0.05,null,Credit Card,Bangalore,Returned,2026-11-10T13:30:01.911Z
O00323,2025-01-04,C0152,P210,Air Conditioner,ELECTRONICS,2,75261.00,0.1,135469.80000000002,Debit Card,Mumbai,completed,2026-11-10T13:30:01.911Z
O00329,2025-01-05,C0449,P205,Levi Jeans,ELECTRONICS,1,null,0.05,null,Debit Card,Kolkata,Cancelled,2026-11-10T13:30:01.911Z


STEP 2: Create Customer Dimension

In [0]:

dim_customer = customers_df.select("customer_id",
                    "customer_name",
                    "email",
                    "city",
                    "state",
                    "loyalty_points",
                    "status"                    
                    )
dim_customer.display()

customer_id,customer_name,email,city,state,loyalty_points,status
C0001,Neha Sharma,nehasharma@email.com,Delhi,Delhi,1608,ACTIVE
C0002,Karan Gupta,karangupta@email.com,Jaipur,Rajasthan,null,ACTIVE
C0003,Sneha Verma,snehaverma@email.com,Jaipur,Maharashtra,null,INACTIVE
C0004,Anjali Kapoor,anjalikapoor@email.com,Mumbai,Uttar Pradesh,null,INACTIVE
C0005,Sneha Singh,snehasingh@email.com,Bangalore,West Bengal,null,INACTIVE
C0006,Karan Mehta,karanmehta@email.com,Chennai,West Bengal,569,ACTIVE
C0007,Rohan Singh,rohansingh@email.com,Mumbai,Delhi,null,INACTIVE
C0008,Amit Rao,amitrao@email.com,Kolkata,Telangana,null,ACTIVE
C0009,Rohan Rao,rohanrao@email.com,Bangalore,Rajasthan,1125,INACTIVE
C0010,Sneha Iyer,snehaiyer@email.com,Lucknow,Maharashtra,null,ACTIVE


In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS retail_project.gold')
dim_customer.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_dim_customer')     

STEP 3: Create Fact Sales Table

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

# fact_sales.display()


from pyspark.sql.functions import year , month , dayofmonth

order_enriched = fact_sales.\
    withColumn("Year",year("order_date")).\
    withColumn("Month",month("order_date")).\
    withColumn("Day",dayofmonth("order_date"))

order_enriched.display()

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,unit_price,discount,total_amount,city,state,gender,loyalty_points,Year,Month,Day
O00079,2025-01-29,null,null,P201,iPhone 15,FASHION,1,null,0.1,null,null,null,null,null,2025,1,29
O00099,2025-01-11,C0664,Amit Kapoor,P203,Sony Headphones,FASHION,3,45265.00,0,135795.0,Kolkata,Maharashtra,M,1509,2025,1,11
O00124,2025-01-12,C0354,Rohan Das,P201,iPhone 15,ELECTRONICS,2,null,0.1,null,Bangalore,Tamil Nadu,M,1670,2025,1,12
O00171,2025-01-14,C0500,Karan Mehta,P205,Levi Jeans,HOME & LIVING,2,null,0.1,null,Chennai,West Bengal,M,null,2025,1,14
O00189,2025-02-12,C0345,Meena Sharma,P205,Levi Jeans,HOME & LIVING,1,null,0,null,Mumbai,Maharashtra,M,null,2025,2,12
O00216,2025-01-26,C0792,Arjun Sharma,P209,Office Chair,HOME & LIVING,2,null,0.1,null,Hyderabad,Telangana,F,null,2025,1,26
O00245,2025-02-05,C0792,Arjun Sharma,P201,iPhone 15,ELECTRONICS,2,76887.00,0,153774.0,Hyderabad,Telangana,F,null,2025,2,5
O00254,2025-02-12,C0814,Arjun Singh,P203,Sony Headphones,ELECTRONICS,1,null,0.05,null,Chennai,Telangana,F,null,2025,2,12
O00323,2025-01-04,C0152,null,P210,Air Conditioner,ELECTRONICS,2,75261.00,0.1,135469.80000000002,null,null,null,null,2025,1,4
O00329,2025-01-05,C0449,Sneha Gupta,P205,Levi Jeans,ELECTRONICS,1,null,0.05,null,Chennai,Maharashtra,M,null,2025,1,5


In [0]:
from pyspark.sql.functions import sum, countDistinct, avg, min,max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()
     


product_id,product_name,customer_id,customer_name,category,Year,Month,order_date,gender,city,state,loyalty_points,Total_revenue,Total_quantity,Total_orders,Avg_revenue,Avg_quantity,Avg_loyalty_points,Min_price,Max_price
P210,Air Conditioner,C0917,Arjun Rao,ELECTRONICS,2025,3,2025-03-19,M,Chennai,Delhi,1767,null,2.0,1,null,2.0,1767.0,null,null
P205,Levi Jeans,C0134,Amit Iyer,FASHION,2025,1,2025-01-20,M,Chennai,Maharashtra,null,43974.0,1.0,1,43974.0,1.0,null,43974.00,43974.00
P204,Nike Shoes,C0468,Meena Iyer,ELECTRONICS,2025,1,2025-01-19,M,Chennai,Delhi,774,null,1.0,1,null,1.0,774.0,null,null
P203,Sony Headphones,C0816,Rohan Sharma,FASHION,2025,2,2025-02-19,M,Chennai,West Bengal,null,88800.3,2.0,1,88800.3,2.0,null,46737.00,46737.00
P201,iPhone 15,C0051,Arjun Das,HOME & LIVING,2025,1,2025-01-23,M,Mumbai,Tamil Nadu,null,208966.5,3.0,1,208966.5,3.0,null,77395.00,77395.00
P204,Nike Shoes,C0460,Amit Rao,ELECTRONICS,2025,3,2025-03-26,F,Mumbai,Delhi,636,null,1.0,1,null,1.0,636.0,null,null
null,HP Laptop,C0645,Arjun Iyer,HOME & LIVING,2025,3,2025-03-19,M,Mumbai,Karnataka,null,null,3.0,1,null,3.0,null,null,null
P201,iPhone 15,C0762,Sneha Malhotra,ELECTRONICS,2025,2,2025-02-09,M,Hyderabad,Delhi,null,157662.0,3.0,1,157662.0,3.0,null,55320.00,55320.00
P210,Air Conditioner,C0904,Sneha Sharma,FASHION,2025,2,2025-02-11,F,Bangalore,Telangana,null,null,3.0,1,null,3.0,null,null,null
P205,Levi Jeans,C0382,null,ELECTRONICS,2025,2,2025-02-26,null,null,null,null,10633.35,3.0,1,10633.35,3.0,null,3731.00,3731.00


In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_fact_Sales')

In [0]:
%sql
select count(*) from retail_project.gold.gold_fact_Sales

count(*)
7480
